### Phase transitions in CNNs

We explore how the Hessian eigenspectrum qualtiatively changes between a minima and non-minima point.

In [1]:
# Libraries import

from multiprocessing import freeze_support

import os
import sys
import pickle
import pprint
from pathlib import Path
import warnings
import numpy as np
import pandas as pd

sys.path.append("../")

import torch as t
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, random_split

from tqdm import tqdm
from datetime import datetime
import json
import wandb

from devinterp.slt import estimate_learning_coeff, estimate_learning_coeff_with_summary
from devinterp.optim.sgld import SGLD
from devinterp.slt import sample
from devinterp.slt.llc import OnlineLLCEstimator
from devinterp.slt.wbic import OnlineWBICEstimator

from approxngd import KFAC
from PyHessian.pyhessian import *
from PyHessian.density_plot import *
from general_utils import *
from hessian_utils import *
from networks import *
from data.build_data import build_data

import plotly.express as px
import plotly.graph_objects as go
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap

device = "cuda" if t.cuda.is_available() else "cpu"
print(device)

c:\Users\moosa\anaconda3\envs\windows_ml\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda


In [32]:
hyperparams = {
    "batch_size" : 128,
    "num_workers" : 12,
    "num_epochs" : 20,
    "lr" : 9e-03,
    "momentum" : 0.9,
}

In [33]:
# Produce a list of CNNs

hidden_conv_layers = 8
models = {}
model = CnnMNIST(hidden_conv_layers=hidden_conv_layers).to(device)
models[0] = model

In [34]:
# load data from pytorch dataloaders

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_set = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_set = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = t.utils.data.DataLoader(train_set, batch_size=hyperparams["batch_size"], shuffle=True, num_workers=hyperparams["num_workers"], persistent_workers=True)
test_loader = t.utils.data.DataLoader(test_set, batch_size=hyperparams["batch_size"], shuffle=False, num_workers=hyperparams["num_workers"], persistent_workers=True)

In [35]:
# training loop (vary number of hidden layers in CNN)

train_losses = []
test_losses = []
print(f"TRAINING WITH {hidden_conv_layers} CONVOLUTION LAYERS")
optimiser = t.optim.SGD(model.parameters(), lr=hyperparams["lr"], momentum=hyperparams["momentum"], nesterov=True)
criterion = nn.CrossEntropyLoss()
for epoch in range(1, hyperparams["num_epochs"]):
    train_loss = train_one_epoch(model, train_loader, optimiser, criterion, device)
    test_loss = evaluate(model, test_loader, criterion, device)
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    models[epoch] = model
    print(f"Epoch {epoch+1}/{hyperparams['num_epochs']}: train_loss={train_loss:.4f}, test_loss={test_loss:.4f}")

TRAINING WITH 8 CONVOLUTION LAYERS


100%|██████████| 469/469 [00:13<00:00, 34.03it/s]


Epoch 2/20: train_loss=2.3017, test_loss=2.3013


100%|██████████| 469/469 [00:06<00:00, 72.96it/s]


Epoch 3/20: train_loss=2.3014, test_loss=2.3012


...

Epoch 18/20: train_loss=0.0836, test_loss=0.0784


100%|██████████| 469/469 [00:06<00:00, 72.26it/s]


Epoch 19/20: train_loss=0.0601, test_loss=0.0525


100%|██████████| 469/469 [00:06<00:00, 71.86it/s]


Epoch 20/20: train_loss=0.0491, test_loss=0.0545


In [36]:
# visualise training and testing data over time

epochs = np.arange(1, hyperparams["num_epochs"]+1)
train_test_fig = go.Figure()
train_test_fig.add_trace(go.Scatter(x=epochs, y=train_losses, mode='lines+markers', name='Train'))
train_test_fig.add_trace(go.Scatter(x=epochs, y=test_losses, mode='lines+markers', name='Test'))
train_test_fig.update_layout(title="Training loss",
                  xaxis_title="Epoch",
                  yaxis_title="Loss",
                  legend_title="Hidden conv layers"
                  )
train_test_fig.show()

In [37]:
# generate hessians
hessians = produce_hessians(models=models,
                            data_loader=test_loader,
                            num_batches=10,
                            criterion={"general" : criterion},
                            device=device)

c:\Users\moosa\anaconda3\envs\windows_ml\lib\site-packages\torch\autograd\__init__.py:266: UserWarning:

Using backward() with create_graph=True will create a reference cycle between the parameter and its gradient which can cause a memory leak. We recommend using autograd.grad when creating the graph to avoid this. If you have to use this function, make sure to reset the .grad fields of your parameters to None after use to break the cycle and avoid the leak. (Triggered internally at ..\torch\csrc\autograd\engine.cpp:1182.)



OutOfMemoryError: CUDA out of memory. Tried to allocate 62.00 MiB. GPU 0 has a total capacity of 4.00 GiB of which 0 bytes is free. Of the allocated memory 2.60 GiB is allocated by PyTorch, and 91.67 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [9]:
# generate data and figures for eigenspectrum
figs, eigenspectrum_data = produce_hessian_eigenspectra(hessians, plot_type="log")

c:\Users\moosa\OneDrive\Documents\windows_dev\ngd_with_slt\notebooks\..\PyHessian\density_plot.py:62: ComplexWarning:

Casting complex values to real discards the imaginary part



In [10]:
for fig in figs:
    fig.show()

In [17]:
figs.append(train_test_fig)

In [21]:
curr_time = datetime.now().strftime("%Y-%m-%d-%H-%M")
write_figs_to_html(figs, f"../cnn_experiments/cnn_hidden_layers_{curr_time}.html", title="Investigating effect of hidden conv layers on CNN Hessian eigenspectrum")